# 1단계(v2): YAMNet 임베딩 추출 + 확인용 베이스라인
(낙상 위상 문제를 고친 새 데이터셋 `4_dataset_v2.zip` 기준)

위에서부터 **순서대로** 셀을 하나씩 실행하세요. (런타임 > 런타임 유형 변경 > T4 GPU 권장, CPU도 가능)

하는 일
1. 구글 드라이브의 `4_dataset_v2.zip`을 찾아서 푼다
2. YAMNet으로 모든 클립의 임베딩을 뽑아 드라이브에 저장한다 (한 번만 하면 됨)
3. 임베딩이 제대로 나왔는지 확인용으로 간단한 분류기를 한 번 학습해 본다

끝나면 **5번, 6번, 7번 셀의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결
실행하면 구글 계정 허용 창이 뜹니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. zip 찾기 · 풀기
드라이브 안 어디에 올렸든 `4_dataset_v2.zip`을 자동으로 찾습니다. 2~3분 걸릴 수 있어요.

In [ ]:
import glob, os, shutil, zipfile

ZIP_PATH = ""  # 자동으로 못 찾을 때만 직접 입력. 예: "/content/drive/MyDrive/AI_HUMAN_CARE/4_dataset_v2.zip"

if not ZIP_PATH:
    found = glob.glob("/content/drive/MyDrive/**/4_dataset_v2.zip", recursive=True)
    print("찾은 파일:", found)
    assert found, "4_dataset_v2.zip을 못 찾았어요. 위의 ZIP_PATH에 직접 입력하세요."
    ZIP_PATH = found[0]

OUT_DIR = os.path.dirname(ZIP_PATH)   # 결과(임베딩)는 zip과 같은 폴더에 저장됩니다
ROOT = "/content/4_dataset_v2"
if not os.path.exists(f"{ROOT}/dataset.csv"):
    shutil.copy(ZIP_PATH, "/content/4_dataset_v2.zip")   # 드라이브에서 바로 풀면 느려서 먼저 복사
    with zipfile.ZipFile("/content/4_dataset_v2.zip") as z:
        z.extractall(ROOT)
print(os.listdir(ROOT))

## 2. 목록 확인
클립 수가 `rebuild_v2.py` 마지막 단계(데이터셋 정리)에서 출력된 Training/Validation 개수와 같은지 확인하세요.

In [ ]:
import pandas as pd
df = pd.read_csv(f"{ROOT}/dataset.csv", encoding="utf-8-sig")
print(len(df), "클립")
print(df.groupby(["split", "label"]).size())

## 3. YAMNet 불러오기

In [ ]:
import csv
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
with tf.io.gfile.GFile(yamnet.class_map_path().numpy()) as f:
    class_names = [row["display_name"] for row in csv.DictReader(f)]
SPEECH = class_names.index("Speech")
print(len(class_names), "클래스 / Speech 번호:", SPEECH)

## 4. 임베딩 추출 (오래 걸리는 셀)
클립 1만 4천 개를 처리합니다. 진행 막대가 끝까지 가면 완료이고, 결과는 드라이브에 저장돼서 다음부터는 다시 안 해도 됩니다.

클립마다 음량을 무작위(-35~-15 dBFS)로 맞춰서 뽑습니다. 위급 클립이 정상 클립보다 8~12dB 컸기 때문에, 모델이 "큰 소리 = 위급"으로 외우는 걸 막기 위한 처리입니다.

In [ ]:
import numpy as np
from scipy.io import wavfile
from tqdm.auto import tqdm

GAIN_RANGE = (-35.0, -15.0)   # 클립마다 목표 음량(dBFS)을 이 범위에서 무작위로 정함
rng = np.random.default_rng(42)

emb_parts, speech_parts, idx_parts = [], [], []
gain_db = np.zeros(len(df), dtype=np.float32)

for i, path in enumerate(tqdm(df["path"].tolist())):
    sr, x = wavfile.read(f"{ROOT}/{path}")
    assert sr == 16000 and x.ndim == 1, path
    x = x.astype(np.float32) / 32768.0
    rms = float(np.sqrt(np.mean(x ** 2))) + 1e-9
    g = min(10 ** (rng.uniform(*GAIN_RANGE) / 20) / rms, 0.99 / (float(np.abs(x).max()) + 1e-9))
    x = x * g
    gain_db[i] = 20 * np.log10(g)
    scores, emb, _ = yamnet(tf.constant(x))
    emb_parts.append(emb.numpy().astype(np.float16))
    speech_parts.append(scores.numpy()[:, SPEECH].astype(np.float16))
    idx_parts.append(np.full(len(emb), i, dtype=np.int32))

X = np.concatenate(emb_parts)
S = np.concatenate(speech_parts)
clip_idx = np.concatenate(idx_parts)
df["gain_db"] = gain_db
np.savez_compressed(f"{OUT_DIR}/embeddings_v2.npz", X=X, speech=S, clip_idx=clip_idx)
df.to_csv(f"{OUT_DIR}/dataset_v2.csv", index=False, encoding="utf-8-sig")
print("프레임 임베딩:", X.shape, "| 저장 위치:", OUT_DIR)

## 5. 임베딩 확인
클립당 프레임 수가 라벨에 따라 얼마나 다른지 봅니다. (위급은 대부분 6프레임 안팎이고 1초짜리 짧은 클립은 1프레임, 정상은 10초라 19프레임이 정상)

In [ ]:
frames_per_clip = np.bincount(clip_idx, minlength=len(df))
df["frames"] = frames_per_clip
print(df.groupby("label")["frames"].describe()[["count", "mean", "min", "max"]].round(1))

## 6. 정상 클립에도 말소리가 있나? (확인용)
YAMNet이 클립마다 "Speech(말소리)" 점수를 매깁니다. 위급은 높고 정상은 낮다면, 모델이 "위급"이 아니라 "말소리"를 배울 수 있다는 뜻입니다.

In [ ]:
frame_df = pd.DataFrame({"clip": clip_idx, "speech": S.astype(np.float32)})
df["speech_max"] = frame_df.groupby("clip")["speech"].max().reindex(range(len(df))).values
print(df.groupby("class")["speech_max"].agg(평균="mean", 말소리비율=lambda s: (s > 0.5).mean()).round(2))

## 7. 확인용 베이스라인
임베딩을 클립별로 평균 내서 가장 단순한 분류기(로지스틱 회귀)를 학습하고, Validation으로 평가합니다.
점수가 아주 높게 나와도 **좋은 모델이라는 뜻이 아닙니다.** 정상 데이터에 일상 대화가 없어서 쉬운 문제이기 때문이고, 그 점을 확인하려는 셀입니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

starts = np.r_[0, np.flatnonzero(np.diff(clip_idx)) + 1]
assert len(starts) == len(df), "클립 수와 임베딩 묶음 수가 달라요"
M = np.add.reduceat(X.astype(np.float32), starts, axis=0) / np.diff(np.r_[starts, len(X)])[:, None]

y = (df["label"] == "urgent").astype(int).values
tr = (df["split"] == "train").values
va = ~tr

clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(M[tr], y[tr])
pred = clf.predict(M[va])
print(classification_report(y[va], pred, target_names=["normal", "urgent"], digits=3))

va_df = df[va].assign(correct=(pred == y[va]))
print("클래스별 맞힌 비율 (Validation):")
print(va_df.groupby("class")["correct"].mean().round(3))